In [77]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, faithfulness
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from langchain_anthropic import ChatAnthropic
import pandas as pd
import copy
import numpy as np
from dotenv import load_dotenv
import os
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [78]:
# Load environment files and store API keys
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_KEY")

In [79]:
# Read in dialogue data
file_path = "../../data/Hallucination/dialogue_data.json"
dialogue_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/general_data.json"
general_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/qa_data.json"
qa_data = pd.read_json(file_path, lines=True)

In [80]:
# Generate a random choice (True for right_answer, False for hallucinated_answer)
choice = np.random.rand(len(qa_data)) < 0.5

# Assign the chosen answer
qa_data["selected_answer"] = np.where(choice, qa_data["right_answer"], qa_data["hallucinated_answer"])

# Add a column indicating the source of the answer
qa_data["hallucinated_flag"] = np.where(choice, 0, 1)

In [81]:
# Set up Claude as the evaluator LLM
claude_llm = ChatAnthropic(api_key=anthropic_key, model="claude-3-7-sonnet-20250219")
evaluator_llm = LangchainLLMWrapper(claude_llm)

In [ ]:
qa_data_ragas = copy.deepcopy(qa_data[["question", "knowledge", "selected_answer"]])

qa_data_ragas = qa_data_ragas.rename(columns=
    {"question": "user_input", 
     "knowledge": "retrieved_contexts", 
     "selected_answer":"response"
     })

qa_data_ragas['retrieved_contexts'] = qa_data_ragas['retrieved_contexts'].map(lambda x: [x])

qa_data_ragas_dict = qa_data_ragas.to_dict(orient='list')

dataset = Dataset.from_dict(qa_data_ragas_dict)

In [ ]:
score = evaluate(dataset,metrics=[faithfulness], llm=evaluator_llm)
qa_results = score.to_pandas()
qa_results.to_csv("../../results/Hallucination/Ragas/qa_results.csv")

In [82]:
# Function to query the LLM with hallucination-aware prompt
async def evaluate_hallucination(context, question, response):
    
    from ragas.dataset_schema import SingleTurnSample 
    from ragas.metrics import Faithfulness

    sample = SingleTurnSample(
            user_input=question,
            response=response,
            retrieved_contexts=context
        )
    scorer = Faithfulness(llm=evaluator_llm)
    faithfulness_score = await scorer.single_turn_ascore(sample)
    return faithfulness_score

In [ ]:
import asyncio

# Initialize list to hold results
results = []

# Define an async function to run all evaluations
async def evaluate_all():
    tasks = []
    
    for index, row in qa_data.head(200).iterrows():
        context = [row.knowledge]
        question = row.question
        response = row.selected_answer
        
        
        # Store async tasks
        tasks.append(
            evaluate_hallucination(context, question, response)
        )

    # Run all evaluations concurrently
    faithfulness_scores = await asyncio.gather(*tasks)
    
    return faithfulness_scores

# Run the async function properly
faithfulness_scores = asyncio.run(evaluate_all())

ValueError: array length 200 does not match index length 10000

In [84]:
qa_results = pd.DataFrame({
    'context': qa_data.head(200).knowledge,
    'prompt': qa_data.head(200).question,
    'response': qa_data.head(200).selected_answer,
    'hallucinated_flag': qa_data.head(200).hallucinated_flag,
    'faithfulness_score': faithfulness_scores,
    'hallucination_score': [1 - i for i in faithfulness_scores]
})

In [85]:
qa_results

,context,prompt,response,hallucinated_flag,faithfulness_score,hallucination_score
0,Arthur's Magazine (1844–1846) was an American ...,Which magazine was started first Arthur's Maga...,First for Women was started first.,1,0.000000,1.000000
1,The Oberoi family is an Indian family that is ...,The Oberoi family is part of a hotel company t...,Delhi,0,1.000000,0.000000
2,"Allison Beth ""Allie"" Goertz (born March 2, 199...",Musician and satirist Allie Goertz wrote a son...,"Allie Goertz wrote a song about Milhouse, a po...",1,0.333333,0.666667
3,"Margaret ""Peggy"" Seeger (born June 17, 1935) i...",What nationality was James Henry Miller's wife?,James Henry Miller's wife was British.,1,0.000000,1.000000
4,It is a hygroscopic solid that is highly solu...,Cadmium Chloride is slightly soluble in this c...,water with a hint of alcohol,1,0.000000,1.000000
...,...,...,...,...,...,...
195,St James Street is a historic street in the to...,St James Street appears as a segment of Whitec...,Stuart period,0,1.000000,0.000000
196,"The Pineground Bridge, also known as the Depot...",The Pineground Bridge formerly carried Depot R...,"2,523",0,1.000000,0.000000
197,Port of Morrow is the fourth studio album by A...,"Which city is the American rock band, that rel...","Albuquerque, New Mexico",0,1.000000,0.000000
198,The Rossendale Free Press is a weekly newspape...,The Rossendale Free Press serves the town how ...,The Rossendale Free Press serves several towns...,1,1.000000,0.000000


In [ ]:
qa_results = pd.read_csv('../../results/Hallucination/Ragas/qa_results.csv')
qa_results['hallucination_score'] = 1 - qa_results.faithfulness

In [86]:
qa_results['hallucination_pred'] = np.where(qa_results['hallucination_score'] > 0.5, 1, 0)
qa_results['hallucinated_flag'] = qa_data.hallucinated_flag

In [87]:
# Compute metrics
accuracy = accuracy_score(qa_results["hallucinated_flag"], qa_results["hallucination_pred"])
precision = precision_score(qa_results["hallucinated_flag"], qa_results["hallucination_pred"])
recall = recall_score(qa_results["hallucinated_flag"], qa_results["hallucination_pred"])

# Print results
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall: {recall*100:.2f}%")

Accuracy: 67.50%
Precision: 75.64%
Recall: 56.19%
